In [ ]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import time

# Define the base URL for Sephora brands and products
base_url = "https://www.sephora.com"

# Headers to mimic a browser request
headers = {
    "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/85.0.4183.121 Safari/537.36"
}

# Data storage
all_reviews = []

def get_brands():
    """
    Scrapes all brand links from the brand listing page.
    Returns a list of URLs for each brand.
    """
    brand_list_url = f"{base_url}/brands-list"
    response = requests.get(brand_list_url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    # Find brand URLs (Inspect the site structure to get the correct tags and classes)
    brands = soup.find_all("a", class_="css-17navuo eanm77i0")
    brand_urls = [base_url + brand['href'] for brand in brands if 'href' in brand.attrs]

    return brand_urls

def get_products(brand_url):
    """
    Given a brand URL, scrape all product links for that brand.
    Returns a list of URLs for each product.
    """
    response = requests.get(brand_url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    # Find product URLs (Inspect the site structure to get the correct tags and classes)
    products = soup.find_all("a", class_="css-klx76")
    product_urls = [base_url + product['href'] for product in products if 'href' in product.attrs]

    return product_urls

def get_reviews(product_url):
    """
    Given a product URL, scrape all reviews and ratings for that product.
    Returns a list of dictionaries with review details.
    """
    response = requests.get(product_url, headers=headers)
    soup = BeautifulSoup(response.text, "html.parser")

    reviews = []

    # Locate each main review block based on your provided structure
    review_blocks = soup.find_all("div", class_="css-c95msn eanm77i0")  # Class for review text

    for review in review_blocks:
        try:
            # Extract the rating from the `aria-label` attribute within a span element
            rating_span = review.find("span", {"aria-label": True})
            rating = rating_span["aria-label"] if rating_span else "No rating"

            # Extract review text from the `css-c95msn eanm77i0` class
            review_text = review.get_text(strip=True)

            # Append the extracted details to the list
            reviews.append({"product_url": product_url, "rating": rating, "review_text": review_text})
        except AttributeError:
            # Skip any reviews that don't match the expected structure
            continue

    return reviews

def main():
    # Step 1: Get all brands
    brand_urls = get_brands()
    print(f"Found {len(brand_urls)} brands.")

    # Step 2: For each brand, get products
    for brand_url in brand_urls:
        print(f"Scraping products for brand: {brand_url}")
        product_urls = get_products(brand_url)

        # Step 3: For each product, get reviews
        for product_url in product_urls:
            print(f"Scraping reviews for product: {product_url}")
            product_reviews = get_reviews(product_url)
            all_reviews.extend(product_reviews)

            # Add delay to prevent rate-limiting
            time.sleep(2)

    # Convert the data to a DataFrame
    df = pd.DataFrame(all_reviews)

    # Save to Excel
    df.to_excel("all_brands_reviews.xlsx", index=False)
    print("Scraping complete. Data saved to all_brands_reviews.xlsx.")

In [ ]:
# Webscrapping takes forever, we can use the pre-downloaded data 'sephora_product_data_cleaned.csv'.
# if __name__ == "__main__":
#     main()

file_path = 'sephora_product_data_cleaned.csv'